# 单因子 IC / 快速分层（OPEN0935_1000 Label）

与 `pipeline_train_score_backtest` 使用同一套 `ic_analysis` + `quick_backtest`。

1. **IC 总表**（全部 `outputs/factors/*.fea`，较快）→ `factor_ic_scan_OPEN0935_1000.csv`
2. 按 `ic_OOS_mean` 等排序后，对**少数因子**生成与 pipeline 相同结构的 `ic_val_to_end/`、`quick_val_to_end/` 图与 CSV。

In [1]:
import sys
sys.path.insert(0, "/root/autodl-tmp")

import pandas as pd
from pathlib import Path

from Strategy import config
from Strategy.label.label_generator import load_label
from Strategy.backtest.single_factor_research import (
    build_reference_tradeable_mask,
    ic_scan_dataframe,
    load_all_factors_glob,
    export_factor_report_discrete_ic_then_bt,
)

LABEL_TAG = "OPEN0935_1000"
BT_PRICE_TAG = "OPEN0935_1000"

OUT = config.BT_RESULT_DIR / "single_factor_study" / f"compare_{LABEL_TAG}"
OUT.mkdir(parents=True, exist_ok=True)

TOP_KS = (20, 50, 100)
TAIL_KS = (20, 50, 100)

label_wide = load_label(LABEL_TAG)
factors = load_all_factors_glob()
print("Label:", label_wide.shape, "| 因子个数:", len(factors))

Label: (1261, 5380) | 因子个数: 134


## 1) 可交易池 mask（与 pipeline 一致，推荐）

用 **任意** 与全市场覆盖度接近的打分宽表作为 `reference_score`（例如 `SCORE_xgb_*.fea` 在 `pipeline_holdout` 下的全区间表）。  
若文件路径不同，请改下面 `ref_path`。

若置 `mask = None`，则 IC 不对可交易池过滤（与 `run_ic_analysis(..., tradeable_mask=None)` 一致，口径更宽）。

In [2]:
ref_path = config.SCORE_OUTPUT_DIR / "pipeline_holdout" / f"SCORE_xgb_{LABEL_TAG}.fea"
if ref_path.exists():
    ref = pd.read_feather(ref_path).set_index("TRADE_DATE")
    mask = build_reference_tradeable_mask(ref, twap_tag=BT_PRICE_TAG)
    print("使用参考打分建 mask:", ref_path, mask.shape)
else:
    mask = None
    print("未找到参考打分, mask=None:", ref_path)

使用参考打分建 mask: /root/autodl-tmp/Strategy/outputs/scores/pipeline_holdout/SCORE_xgb_OPEN0935_1000.fea (1251, 5324)


## 2) 全因子 IC 扫描（无图，写 CSV）

In [3]:
scan_path = OUT / f"factor_ic_scan_{LABEL_TAG}.csv"
ic_tab = ic_scan_dataframe(label_wide, factors, tradeable_mask=mask, min_stocks=10)
ic_tab.to_csv(scan_path, index=False)
print("已保存:", scan_path, "shape=", ic_tab.shape)

col_oos = [c for c in ic_tab.columns if c.startswith("ic_OOS") and c.endswith("mean")]
if col_oos:
    ic_tab = ic_tab.sort_values(col_oos[0], ascending=False)
print(ic_tab.head(20).to_string())

IC scan (factors):   0%|          | 0/134 [00:00<?, ?it/s]

/root/autodl-tmp/Strategy/backtest/ic_analysis.py:81: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask = tradeable_mask.reindex(index=common_dates, columns=common_stocks).fillna(False)
/root/autodl-tmp/Strategy/backtest/ic_analysis.py:81: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask = tradeable_mask.reindex(index=common_dates, columns=common_stocks).fillna(False)
/root/autodl-tmp/Strategy/backtest/ic_analysis.py:81: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call re

已保存: /root/autodl-tmp/Strategy/outputs/bt_results/single_factor_study/compare_OPEN0935_1000/factor_ic_scan_OPEN0935_1000.csv shape= (134, 41)
                 factor  ic_Train_n_days  ic_Train_mean  ic_Train_std  ic_Train_icir  ic_Train_ic_win_rate  ic_Val_n_days  ic_Val_mean  ic_Val_std  ic_Val_icir  ic_Val_ic_win_rate  ic_OOS_n_days  ic_OOS_mean  ic_OOS_std  ic_OOS_icir  ic_OOS_ic_win_rate  ic_Overall_n_days  ic_Overall_mean  ic_Overall_std  ic_Overall_icir  ic_Overall_ic_win_rate  rank_ic_Train_n_days  rank_ic_Train_mean  rank_ic_Train_std  rank_ic_Train_icir  rank_ic_Train_ic_win_rate  rank_ic_Val_n_days  rank_ic_Val_mean  rank_ic_Val_std  rank_ic_Val_icir  rank_ic_Val_ic_win_rate  rank_ic_OOS_n_days  rank_ic_OOS_mean  rank_ic_OOS_std  rank_ic_OOS_icir  rank_ic_OOS_ic_win_rate  rank_ic_Overall_n_days  rank_ic_Overall_mean  rank_ic_Overall_std  rank_ic_Overall_icir  rank_ic_Overall_ic_win_rate
108      close_location            637.0       0.009002      0.085683       0.105059      

## 3) 对 Top-N 或指定因子名出图（与 compare_OPEN0935_1000/xgb 目录结构类似）

每个因子子目录: `OUT/<factor名>/ic_val_to_end/`、`quick_val_to_end/`。

In [4]:
N_PLOT = 5   # 只画 IC 排名前 N 个（全量因子会极慢）
extra = []   # 可写死关心因子名，如 ["ret_std_20d", "momentum_20d"]

rank_col = "ic_OOS_mean" if "ic_OOS_mean" in ic_tab.columns else ic_tab.columns[0]
names = list(ic_tab.sort_values(rank_col, ascending=False).head(N_PLOT)["factor"])
for e in extra:
    if e in factors and e not in names:
        names.append(e)

for name in names:
    print("---", name, "---")
    export_factor_report_discrete_ic_then_bt(
        name,
        factors[name],
        label_wide,
        OUT,
        tradeable_mask=mask,
        val_start=config.VAL_START,
        end_date=None,
        top_ks=TOP_KS,
        tail_ks=TAIL_KS,
        twap_tag=BT_PRICE_TAG,
    )
print("输出根目录:", OUT)

--- close_location ---


/root/autodl-tmp/Strategy/backtest/ic_analysis.py:81: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask = tradeable_mask.reindex(index=common_dates, columns=common_stocks).fillna(False)
/root/autodl-tmp/Strategy/backtest/ic_analysis.py:319: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


--- gap_ratio ---


/root/autodl-tmp/Strategy/backtest/ic_analysis.py:81: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask = tradeable_mask.reindex(index=common_dates, columns=common_stocks).fillna(False)
/root/autodl-tmp/Strategy/backtest/ic_analysis.py:319: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


--- cum_mmd_60 ---


...

--- candle_skew ---


/root/autodl-tmp/Strategy/backtest/ic_analysis.py:81: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask = tradeable_mask.reindex(index=common_dates, columns=common_stocks).fillna(False)
/root/autodl-tmp/Strategy/backtest/ic_analysis.py:319: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


--- atr_ratio_20 ---


/root/autodl-tmp/Strategy/backtest/ic_analysis.py:81: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask = tradeable_mask.reindex(index=common_dates, columns=common_stocks).fillna(False)
/root/autodl-tmp/Strategy/backtest/ic_analysis.py:319: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


输出根目录: /root/autodl-tmp/Strategy/outputs/bt_results/single_factor_study/compare_OPEN0935_1000
